In [11]:
pip install db-dtypes

  Using cached db_dtypes-1.5.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached pandas-2.3.3-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached pyarrow-24.0.0-cp314-cp314-win_amd64.whl.metadata (3.0 kB)
  Using cached pytz-2026.2-py2.py3-none-any.whl.metadata (22 kB)
Using cached db_dtypes-1.5.1-py3-none-any.whl (18 kB)
Using cached pandas-2.3.3-cp314-cp314-win_amd64.whl (11.1 MB)
Using cached pyarrow-24.0.0-cp314-cp314-win_amd64.whl (28.0 MB)
Using cached pytz-2026.2-py2.py3-none-any.whl (510 kB)

   ---------------------------------------- 0/4 [pytz]
   ---------------------------------------- 0/4 [pytz]
   ---------- ----------------------------- 1/4 [pyarrow]
   ---------- ----------------------------- 1/4 [pyarrow]
   ---------- ----------------------------- 1/4 [pyarrow]
   ---------- ----------------------------- 1/4 [pyarrow]
   ---------- ----------------------------- 1/4 [pyarrow]
   ---------- ----------------------------- 1/4 [pyarrow]
   ---------- ----------------

  You can safely remove it manually.

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\abbal\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [1]:
!pip install google-cloud-bigquery pandas db-dtypes pyarrow

  Using cached google_cloud_bigquery-3.41.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached google_api_core-2.30.3-py3-none-any.whl.metadata (3.1 kB)
  Using cached google_auth-2.50.0-py3-none-any.whl.metadata (6.2 kB)
  Using cached google_cloud_core-2.5.1-py3-none-any.whl.metadata (2.8 kB)
  Using cached google_resumable_media-2.8.2-py3-none-any.whl.metadata (2.2 kB)
  Using cached googleapis_common_protos-1.74.0-py3-none-any.whl.metadata (9.2 kB)
  Using cached proto_plus-1.27.2-py3-none-any.whl.metadata (2.2 kB)
  Using cached grpcio-1.80.0-cp314-cp314-win_amd64.whl.metadata (3.9 kB)
  Using cached grpcio_status-1.80.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached cryptography-48.0.0-cp311-abi3-win_amd64.whl.metadata (4.3 kB)
  Using cached google_crc32c-1.8.0-cp314-cp314-win_amd64.whl.metadata (1.8 kB)
Using cached google_cloud_bigquery-3.41.0-py3-none-any.whl (262 kB)
Using cached google_api_core-2.30.3-


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\abbal\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [ ]:
# =========================
# 2. IMPORTS Y CONEXIÓN
# =========================
from google.cloud import bigquery
from google.oauth2 import service_account

PROJECT_ID = "tc-sql-jointech"
DATASET_ID = "jointech"
CREDENTIALS_PATH = "../credentials/service-account.json"

credentials = service_account.Credentials.from_service_account_file(
    CREDENTIALS_PATH
)

client = bigquery.Client(
    project=PROJECT_ID,
    credentials=credentials
)



print("✅ Conectado correctamente a BigQuery")


✅ Conectado correctamente a BigQuery


In [8]:
def run(query):
    return client.query(query).to_dataframe()

In [10]:
# =========================
# 3. VERIFICAR TABLAS
# =========================

query = f"""
SELECT table_name
FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.TABLES`
"""

run(query)


ValueError: Please install the 'db-dtypes' package to use this function.

In [3]:
# =========================
# 4. VALIDACIÓN DE VOLUMEN
# =========================

query = """
SELECT 'customers' AS table_name, COUNT(*) AS rows FROM `tc-sql-jointech.jointech.customers`
UNION ALL
SELECT 'categories', COUNT(*) FROM `tc-sql-jointech.jointech.categories`
UNION ALL
SELECT 'products', COUNT(*) FROM `tc-sql-jointech.jointech.products`
UNION ALL
SELECT 'orders', COUNT(*) FROM `tc-sql-jointech.jointech.orders`
UNION ALL
SELECT 'order_items', COUNT(*) FROM `tc-sql-jointech.jointech.order_items`
UNION ALL
SELECT 'payments', COUNT(*) FROM `tc-sql-jointech.jointech.payments`
UNION ALL
SELECT 'reviews', COUNT(*) FROM `tc-sql-jointech.jointech.reviews`
ORDER BY rows DESC
"""

df_volume = client.query(query).to_dataframe()

print("📊 VOLUMEN DE DATOS")
display(df_volume)


BadRequest: 400 Syntax error: Unexpected keyword ROWS at [2:47]; reason: invalidQuery, location: query, message: Syntax error: Unexpected keyword ROWS at [2:47]

Location: EU
Job ID: 35b91ad5-c0c9-4c01-8ac6-2ed8e3da2417


In [ ]:
#Ver tablas del dataset
SELECT table_name
FROM `tc-sql-jointech.jointech.INFORMATION_SCHEMA.TABLES`
ORDER BY table_name;

In [ ]:
#Conteo de registros por tabla
SELECT 'customers' AS table_name, COUNT(*) AS rows FROM `tc-sql-jointech.jointech.customers`
UNION ALL
SELECT 'orders', COUNT(*) FROM `tc-sql-jointech.jointech.orders`
UNION ALL
SELECT 'products', COUNT(*) FROM `tc-sql-jointech.jointech.products`
UNION ALL
SELECT 'order_items', COUNT(*) FROM `tc-sql-jointech.jointech.order_items`
UNION ALL
SELECT 'payments', COUNT(*) FROM `tc-sql-jointech.jointech.payments`
UNION ALL
SELECT 'categories', COUNT(*) FROM `tc-sql-jointech.jointech.categories`
UNION ALL
SELECT 'reviews', COUNT(*) FROM `tc-sql-jointech.jointech.reviews`;

In [ ]:
#Integridad customers → orders
SELECT
  c.customer_id,
  COUNT(o.order_id) AS total_orders
FROM `tc-sql-jointech.jointech.customers` c
LEFT JOIN `tc-sql-jointech.jointech.orders` o
  ON c.customer_id = o.customer_id
GROUP BY c.customer_id
ORDER BY total_orders DESC;

In [ ]:
#Orders sin customer (ERROR)
SELECT *
FROM `tc-sql-jointech.jointech.orders`
WHERE customer_id IS NULL;

In [ ]:
#Productos sin categoría (ERROR FK)
SELECT p.*
FROM `tc-sql-jointech.jointech.products` p
LEFT JOIN `tc-sql-jointech.jointech.categories` c
  ON p.category_id = c.category_id
WHERE c.category_id IS NULL;

In [ ]:
Ingresos 
SELECT SUM(amount) AS total_revenue
FROM `tc-sql-jointech.jointech.payments`
WHERE payment_status = 'completed';